# COMM117 Week 7/8 Workshop — Translation + RAG

**How to use:** complete the TODO cells first, then run the quick checks.

This workshop combines the main ideas from:
- **Lecture 15:** translation with prompting, constraints, evaluation, and when fine-tuning wins
- **Lecture 16:** retrieval-augmented generation (RAG), chunking, dense vs lexical retrieval, reranking, and hybrid search




## Part 0 — Setup (do not edit)

We use **tiny toy examples** so everything runs quickly on CPU.


In [ ]:

import re
import math
from collections import Counter, defaultdict

print("Combined Translation + RAG workshop setup ready.")


## Part 1 — Translation prompting and constraint preservation

Lecture 15 emphasized that a good translation prompt often follows:

**Role + Task + Constraints + Output format + Examples**

And that translation quality is not only about fluency — we must often preserve:
- placeholders
- HTML/XML tags
- product names / entities
- numbers / dates / times
- glossary terms


### Exercise 1.1 — Build a constraints-first translation prompt

Write `build_translation_prompt(...)` so it creates a single prompt string that includes:
1. the target language
2. the source text
3. an instruction to preserve numbers and placeholders exactly
4. glossary mappings
5. a strict instruction not to add new information


In [ ]:

# TODO
sample_glossary = {"Account": "Konto", "token": "token"}
source_text = "DataHub will be unavailable from 2:00 to 3:30 PM for {first_name}."
target_lang = "German"

def build_translation_prompt(source_text, target_lang, glossary):
    raise NotImplementedError

prompt = build_translation_prompt(source_text, target_lang, sample_glossary)
print(prompt)


### Exercise 1.2 — Check placeholders, tags, and numbers

Implement helper functions:
- `extract_placeholders(text)` for patterns like `{first_name}`
- `extract_tags(text)` for patterns like `<b>` and `</b>`
- `extract_numbers(text)` for numbers / time-like strings such as `2:00`, `3:30`


In [ ]:

# TODO
def extract_placeholders(text):
    raise NotImplementedError

def extract_tags(text):
    raise NotImplementedError

def extract_numbers(text):
    raise NotImplementedError

src = "Click <b>Save</b> before 2:00 PM, {first_name}."
hyp = "Klicken Sie vor 2:00 PM auf <b>Speichern</b>, {first_name}."

print("placeholders:", extract_placeholders(src), extract_placeholders(hyp))
print("tags:", extract_tags(src), extract_tags(hyp))
print("numbers:", extract_numbers(src), extract_numbers(hyp))


### Exercise 1.3 — Constraint validator

Implement `constraint_report(src, hyp)` returning a dictionary with Boolean fields:
- `placeholders_ok`
- `tags_ok`
- `numbers_ok`

This reflects the lecture point that **translation must often be validated explicitly**, not judged only by fluency.


In [ ]:

# TODO
def constraint_report(src, hyp):
    raise NotImplementedError

src = "DataHub will be unavailable from 2:00 to 3:30 PM for {first_name}."
good = "DataHub ist von 2:00 bis 3:30 PM für {first_name} nicht verfügbar."
bad  = "Data Hub ist heute Nachmittag für den Benutzer nicht verfügbar."

print("good:", constraint_report(src, good))
print("bad :", constraint_report(src, bad))


## Part 2 — Translation evaluation and decision making

Lecture 15 introduced lightweight automatic evaluation:
- BLEU-like lexical overlap
- chrF intuition from character n-grams
- explicit constraint checks
- human review for adequacy / terminology / formatting


### Exercise 2.1 — Tiny unigram precision

Implement a simple unigram precision:

$$
\text{precision} = \frac{\text{matched unigram count}}{\text{candidate unigram count}}
$$

Use clipped counts.


In [ ]:

# TODO
def unigram_precision(reference, candidate):
    raise NotImplementedError

ref = "Klicken Sie auf Speichern"
cand1 = "Klicken Sie auf Speichern"
cand2 = "Bitte klicken Sie auf Speichern jetzt"

print(unigram_precision(ref, cand1))
print(unigram_precision(ref, cand2))


### Exercise 2.2 — Tiny BLEU-1 with brevity penalty

Implement a function equivalent to:

- `bleu1(pred, ref)`  
- unigram precision over whitespace-separated tokens
- **clipped counts**
- brevity penalty  
  \[
  BP = 
  \begin{cases}
  1 & \text{if } c > r \\
  e^{(1-r/c)} & \text{otherwise}
  \end{cases}
  \]
  where `c = len(pred_tokens)` and `r = len(ref_tokens)`

Return:
$$
\text{BLEU-1} = BP \times \text{precision}
$$
More specific guide:

1. Tokenize with `.split()`
2. Count candidate tokens and reference tokens
3. For each candidate token, count at most the number of times it appears in the reference (**clipping**)
4. Precision = `matched_clipped_tokens / len(candidate_tokens)`
5. Compute `BP`
6. Return `BP * precision`

Edge case:
- if the candidate/prediction is empty, return `0.0`

Example:
- `pred = "das ist gut"`
- `ref  = "das ist sehr gut"`
- precision = `3/3`
- `BP = exp(1 - 4/3)`
- BLEU-1 = `BP * 1.0`

This is still a tiny teaching version of BLEU, not the full corpus BLEU metric.


In [ ]:

# TODO
def brevity_penalty(reference, candidate):
    raise NotImplementedError

def tiny_bleu1(reference, candidate):
    raise NotImplementedError

print("BLEU-1 exact-ish:", tiny_bleu1("das ist gut", "das ist gut"))
print("BLEU-1 short:", tiny_bleu1("das ist sehr gut", "das ist gut"))


### Exercise 2.3 — Why lexical overlap is not enough

Fill in the variables below.

Scenario:
- Candidate A has high lexical overlap but breaks HTML tags.
- Candidate B has lower lexical overlap but preserves all tags/placeholders and meaning.

Which candidate should be preferred for production localization?


In [ ]:

# TODO
preferred_candidate = None   # "A" or "B"



## Part 3 — Retrieval basics for RAG

Lecture 16 introduced the RAG pipeline:

**Query → Retrieve → Prompt Template → LLM → Answer**

Key ideas:
- retrieval quality strongly affects answer quality
- chunking quality matters a lot
- lexical and dense retrieval have different strengths
- sometimes hybrid retrieval works best


### Exercise 3.1 — Fixed-size chunking with overlap

Implement `chunk_tokens(tokens, chunk_size, overlap)`.

Example:
- chunk_size = 5
- overlap = 2

Then chunks start like:
- 0:5
- 3:8
- 6:11
...


In [ ]:

# TODO
def chunk_tokens(tokens, chunk_size=5, overlap=2):
    raise NotImplementedError

tokens = "refund policy for digital products updated in january 2026 only original purchaser may request refund".split()
chunks = chunk_tokens(tokens, chunk_size=5, overlap=2)
chunks


### Exercise 3.2 — Simple lexical retrieval score

Implement a tiny lexical score:
- lowercase
- split on words
- score = number of overlapping query terms with the chunk

Return chunks sorted from highest score to lowest score.


In [ ]:

# TODO
def lexical_score(query, chunk):
    raise NotImplementedError

def rank_lexical(query, chunks):
    raise NotImplementedError

chunks = [
    "refund policy for digital products",
    "shipping policy for physical goods",
    "account security and password reset",
    "digital purchases are non refundable unless required by law",
]
query = "digital refund policy"

rank_lexical(query, chunks)


### Exercise 3.3 — Dense retrieval intuition with toy embeddings

We use a tiny hand-made embedding dictionary.
Represent a text as the **sum of token vectors**, then compute cosine similarity.

This is only to build intuition for semantic retrieval.


In [ ]:

toy_vec = {
    "refund": [1.0, 0.0, 0.0],
    "return": [0.9, 0.1, 0.0],
    "policy": [0.0, 1.0, 0.0],
    "rule": [0.0, 0.9, 0.1],
    "digital": [0.0, 0.0, 1.0],
    "download": [0.1, 0.0, 0.9],
    "shipping": [0.2, 0.8, 0.0],
}

# TODO
def embed_text(text, vecs):
    raise NotImplementedError

def cosine(u, v):
    raise NotImplementedError

def rank_dense(query, chunks, vecs):
    raise NotImplementedError

chunks = [
    "refund policy digital",
    "return rule download",
    "shipping policy",
]
query = "digital refund rule"

rank_dense(query, chunks, toy_vec)


## Part 4 — RAG prompt construction and grounding

Lecture 16 stressed that the generator should be constrained by retrieved evidence.

Typical prompt idea:

> Use ONLY the context below.  
> If the answer is not supported, say "Not found."


### Exercise 4.1 — Build a grounded RAG prompt

Implement `build_rag_prompt(query, retrieved_chunks)` that:
- instructs the model to use only the context
- includes the retrieved chunks
- tells the model to say `"Not found"` if unsupported


In [ ]:

# TODO
def build_rag_prompt(query, retrieved_chunks):
    raise NotImplementedError

retrieved = [
    "Policy v3: digital purchases are non-refundable except where local law requires otherwise.",
    "Refund requests must be submitted within 14 days for eligible items."
]
print(build_rag_prompt("Can I refund a digital product?", retrieved))


### Exercise 4.2 — Supported or unsupported?

Implement `is_supported(answer, retrieved_chunks)` with a **very simple heuristic**:
return `True` only if all non-trivial content words in the answer appear somewhere in the retrieved context.



In [ ]:

STOPWORDS = {"the","a","an","is","are","of","to","and","or","for","in","on","if","be","by","with","can","i"}

# TODO
def is_supported(answer, retrieved_chunks):
    raise NotImplementedError

ctx = [
    "digital purchases are non-refundable except where local law requires otherwise",
    "refund requests must be submitted within 14 days for eligible items"
]

ans1 = "Digital purchases are non-refundable except where local law requires otherwise."
ans2 = "All digital purchases can be refunded within 30 days."

print(is_supported(ans1, ctx))
print(is_supported(ans2, ctx))
